In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

print(f"yfinance version: {yf.__version__}")

yfinance version: 1.3.0


In [2]:
ticker_list = ['2267.T', '3435.T', '1846.HK', '6889.HK', '7399.T', '1913.HK', 'VTU.L']


def _to_naive(ts):
    """Safely convert any Timestamp to tz-naive UTC.
    
    FIX A: annual_bs.columns are tz-naive; get_shares_full() index is tz-aware.
    Calling .tz_localize(None) on an already-naive timestamp raises TypeError,
    silently swallowed by except:pass — meaning hist_market_cap is always None.
    Use .tz_convert(None) for aware, leave naive untouched.
    """
    if ts.tzinfo is not None:
        return ts.tz_convert(None)
    return ts


def pull_yf_ticker_data(ticker_list: list):
    """Pull all available annual balance sheet + income data from Yahoo Finance.
    Returns a multi-row DataFrame structured by Ticker and Year.
    """
    extracted_data = []

    for ticker_str in ticker_list:
        print(f"Fetching historical data for: {ticker_str}...")
        try:
            ticker = yf.Ticker(ticker_str)
            info = ticker.info

            # --- Currencies (guarded against None) ---
            raw_trading = info.get("currency", None)
            raw_financial = info.get("financialCurrency", None)
            trading_curr = raw_trading.upper() if raw_trading else None
            financial_curr = raw_financial.upper() if raw_financial else None

            market_cap = info.get("marketCap", None)

            # --- FX rate (default 1.0 = no conversion needed) ---
            fx_rate = 1.0
            if trading_curr and financial_curr and trading_curr != financial_curr:
                fx_ticker_str = f"{trading_curr}{financial_curr}=X"
                try:
                    fx_data = yf.Ticker(fx_ticker_str).history(period="1d")
                    if not fx_data.empty:
                        fx_rate = fx_data["Close"].iloc[-1]
                except Exception:
                    pass

            # FIX B: use `is not None` instead of truthiness check so marketCap=0
            # is handled correctly (rare but possible for shell/holding companies).
            market_cap_converted = (market_cap * fx_rate) if market_cap is not None else None

            # --- Historical shares outstanding ---
            shares_series = None
            try:
                shares_series = ticker.get_shares_full(start="2015-01-01", end=None)
            except Exception:
                pass

            # --- TTM Net Income ---
            ttm_net_inc = None
            try:
                ttm_inc = ticker.ttm_income_stmt
                if isinstance(ttm_inc, tuple):
                    ttm_inc = ttm_inc[0]
                if not isinstance(ttm_inc, pd.DataFrame):
                    ttm_inc = pd.DataFrame(ttm_inc)
                if not ttm_inc.empty and "Net Income" in ttm_inc.index:
                    ttm_net_inc = ttm_inc.loc["Net Income"].iloc[0]
            except Exception:
                pass

            if ttm_net_inc is None:
                try:
                    q_inc = ticker.quarterly_income_stmt
                    if isinstance(q_inc, tuple):
                        q_inc = q_inc[0]
                    if not isinstance(q_inc, pd.DataFrame):
                        q_inc = pd.DataFrame(q_inc)
                    if not q_inc.empty and "Net Income" in q_inc.index:
                        ttm_net_inc = q_inc.loc["Net Income"].iloc[:4].sum()
                except Exception:
                    pass

            # --- Annual statements ---
            raw_bs = ticker.balance_sheet
            raw_inc = ticker.income_stmt
            if isinstance(raw_bs, tuple): raw_bs = raw_bs[0]
            if isinstance(raw_inc, tuple): raw_inc = raw_inc[0]
            annual_bs = raw_bs if isinstance(raw_bs, pd.DataFrame) else pd.DataFrame(raw_bs)
            annual_inc = raw_inc if isinstance(raw_inc, pd.DataFrame) else pd.DataFrame(raw_inc)

            if not annual_bs.empty:
                for report_date in annual_bs.columns:
                    date_str = str(report_date.date())
                    year_val = report_date.year
                    bs_col = annual_bs[report_date]

                    total_assets = bs_col.get("Total Assets", None)
                    current_assets = bs_col.get("Current Assets", None)
                    total_liabilities = bs_col.get("Total Liabilities Net Minority Interest", None)
                    total_goodwill_intangibles = bs_col.get("Goodwill And Other Intangible Assets", None)
                    total_equity = bs_col.get("Common Stock Equity", None)

                    # FIX C: Investment Properties is almost never filed outside REITs.
                    # Fall back to Investments And Advances, which covers long-term
                    # investment holdings across JP/HK/L company filings more reliably.
                    total_investments = bs_col.get(
                        "Investment Properties",
                        bs_col.get("Investments And Advances", None)
                    )

                    # --- Historical Market Cap ---
                    hist_market_cap = None
                    shares_outstanding = None
                    report_date_naive = _to_naive(report_date)  # FIX A applied here

                    # Strategy A: shares from get_shares_full time series
                    if shares_series is not None and not shares_series.empty:
                        try:
                            closest_share_date = min(
                                shares_series.index,
                                key=lambda x: abs(_to_naive(x) - report_date_naive),  # FIX A
                            )
                            if abs((_to_naive(closest_share_date) - report_date_naive).days) <= 180:
                                shares_outstanding = shares_series.loc[closest_share_date]
                        except Exception:
                            pass

                    # Strategy B: balance sheet Ordinary Shares Number fallback
                    if shares_outstanding is None or pd.isna(shares_outstanding):
                        shares_outstanding = bs_col.get("Ordinary Shares Number", None)

                    if shares_outstanding is not None and not pd.isna(shares_outstanding):
                        try:
                            start_search = report_date_naive - pd.Timedelta(days=3)
                            end_search = report_date_naive + pd.Timedelta(days=4)
                            price_hist = ticker.history(start=start_search, end=end_search)

                            if not price_hist.empty:
                                # FIX D: price_hist index is tz-aware; use tz_convert not tz_localize
                                price_hist.index = price_hist.index.map(_to_naive)
                                closest_price_idx = min(
                                    price_hist.index,
                                    key=lambda x: abs(x - report_date_naive),
                                )
                                close_price = price_hist.loc[closest_price_idx, "Close"]
                                hist_market_cap = shares_outstanding * close_price * fx_rate
                        except Exception:
                            pass

                    # --- Match income statement ---
                    fy_net_inc = None
                    if not annual_inc.empty:
                        if report_date in annual_inc.columns:
                            fy_net_inc = annual_inc[report_date].get("Net Income", None)
                        else:
                            closest_col = min(
                                annual_inc.columns,
                                key=lambda x: abs(x - report_date),
                            )
                            if abs((closest_col - report_date).days) <= 7:
                                fy_net_inc = annual_inc[closest_col].get("Net Income", None)

                    is_latest = report_date == annual_bs.columns[0]
                    final_market_cap = market_cap_converted if is_latest else hist_market_cap
                    if is_latest and final_market_cap is None:
                        final_market_cap = hist_market_cap

                    extracted_data.append({
                        "Ticker": ticker_str,
                        "Year": year_val,
                        "Report Date": date_str,
                        "Trading Currency": trading_curr,
                        "Financial Currency": financial_curr,
                        "Market Cap": final_market_cap,   # None kept as None — not 0
                        "Total Assets": total_assets,
                        "Total Current Assets": current_assets,
                        "Total Goodwill and Intangibles": total_goodwill_intangibles,
                        "Total Liabilities": total_liabilities,
                        "Total Equity": total_equity,
                        "Total Investments": total_investments,
                        "Latest FY Net Income": fy_net_inc,
                        "TTM Net Income": ttm_net_inc if is_latest else None,
                    })
            else:
                print(f"   No annual balance sheet found for {ticker_str}")

        except Exception as e:
            print(f"Error fetching data for {ticker_str}: {e}")

    df = pd.DataFrame(extracted_data)
    if not df.empty:
        df = df.sort_values(by=["Ticker", "Year"], ascending=[True, False]).reset_index(drop=True)
    return df

In [3]:
def format_currency(val):
    """Formats large numbers into readable strings. Returns 'N/A' for missing data."""
    # FIX E: Guard against NaN/None — previously crashed when NaN reached this function
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return "N/A"
    abs_val = abs(val)
    if abs_val >= 1_000_000_000_000:
        return f"{val / 1_000_000_000_000:.1f}t"
    elif abs_val >= 1_000_000_000:
        return f"{val / 1_000_000_000:.1f}b"
    elif abs_val >= 1_000_000:
        return f"{val / 1_000_000:.1f}m"
    elif abs_val >= 1_000:
        return f"{val / 1_000:.1f}k"
    return str(int(val)) if val == int(val) else f"{val:.1f}"


def format_ratio(val):
    """Returns 'N/A' for negative, infinite, or NaN ratios."""
    if pd.isna(val) or val < 0 or val in (float("inf"), float("-inf")):
        return "N/A"
    return f"{val:.2f}x"


def format_percentage(val):
    if pd.isna(val) or val in (float("inf"), float("-inf")):
        return "N/A"
    return f"{val:.2%}"

In [4]:
df = pull_yf_ticker_data(ticker_list=ticker_list)


def create_calcs(df_):
    # Sort ascending so rolling windows look backwards in time correctly
    df_ = df_.sort_values(by=["Ticker", "Year"], ascending=[True, True]).copy()

    # FIX F: Remove the blanket fillna(0) — it turns missing Market Cap into 0,
    # making every ratio look like a valid 0.00x instead of N/A.
    # Instead, only fill fields where 0 is a correct default:
    #   - Total Investments / Goodwill: absence = genuinely zero held
    #   - Everything else stays NaN so ratios propagate NaN => 'N/A' after formatting
    df_["Total Investments"] = df_["Total Investments"].fillna(0)
    df_["Total Goodwill and Intangibles"] = df_["Total Goodwill and Intangibles"].fillna(0)

    df_ = df_.assign(
        NCAV=lambda d: d["Total Current Assets"] - d["Total Liabilities"],
        NCAV_Inv=lambda d: d["Total Current Assets"] - d["Total Liabilities"] + d["Total Investments"],
        Book_Value=lambda d: d["Total Assets"] - d["Total Liabilities"],
        Tangible_Book_Value=lambda d: (
            d["Total Assets"] - d["Total Liabilities"] - d["Total Goodwill and Intangibles"]
        ),

        # FIX G: Use Total Equity (Common Stock Equity from balance sheet) for ROE,
        # not Book_Value. Book_Value = Assets - Liabilities includes minority interests
        # and other items that overstate the equity base for ROE purposes.
        # Replace 0 equity with NaN to avoid divide-by-zero producing inf.
        ROE=lambda d: d["Latest FY Net Income"] / d["Total Equity"].replace(0, np.nan),

        # Ratios: NaN Market Cap or NaN/zero denominator => NaN => 'N/A' after format
        Price_Book_Ratio=lambda d: d["Market Cap"] / d["Book_Value"].replace(0, np.nan),
        Price_Tangible_Book_Ratio=lambda d: d["Market Cap"] / d["Tangible_Book_Value"].replace(0, np.nan),
        Price_NCAV_Ratio=lambda d: d["Market Cap"] / d["NCAV"].replace(0, np.nan),
        Price_NCAV_Inv_Ratio=lambda d: d["Market Cap"] / d["NCAV_Inv"].replace(0, np.nan),
        PE_Ratio=lambda d: d["Market Cap"] / d["Latest FY Net Income"].replace(0, np.nan),
        PE_Ratio_TTM=lambda d: d["Market Cap"] / d["TTM Net Income"].replace(0, np.nan),
    )

    # Sort back to descending before formatting
    df_ = df_.sort_values(by=["Ticker", "Year"], ascending=[True, False]).reset_index(drop=True)

    # --- Format currency columns ---
    currency_cols = [
        "Market Cap", "Total Assets", "Total Current Assets", "Total Liabilities",
        "Total Investments", "Total Goodwill and Intangibles", "Total Equity",
        "NCAV", "NCAV_Inv", "Book_Value", "Tangible_Book_Value",
        "Latest FY Net Income", "TTM Net Income",
    ]
    for col in currency_cols:
        if col in df_.columns:
            df_[col] = df_[col].map(format_currency)

    # --- Format ratio columns ---
    ratio_cols = [
        "Price_Book_Ratio", "Price_Tangible_Book_Ratio",
        "Price_NCAV_Ratio", "Price_NCAV_Inv_Ratio",
        "PE_Ratio", "PE_Ratio_TTM",
    ]
    for col in ratio_cols:
        if col in df_.columns:
            df_[col] = df_[col].map(format_ratio)

    # --- Format percentage columns ---
    for col in ["ROE", "ROE_5YR"]:
        if col in df_.columns:
            df_[col] = df_[col].map(format_percentage)

    return df_


df_calculated = create_calcs(df)
df_calculated

Fetching historical data for: 2267.T...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


Error fetching data for 2267.T: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Fetching historical data for: 3435.T...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


Error fetching data for 3435.T: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Fetching historical data for: 1846.HK...


...

Error fetching data for 7399.T: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Fetching historical data for: 1913.HK...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()


Error fetching data for 1913.HK: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().
Fetching historical data for: VTU.L...


c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrapers\history.py:389: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  self._capital_gains = pd.Series()
c:\Users\alexa\AppData\Local\Programs\Python\Python310\lib\site-packages\yfinance\scrape

,Ticker,Year,Report Date,Trading Currency,Financial Currency,Market Cap,Total Assets,Total Current Assets,Total Goodwill and Intangibles,Total Liabilities,...,NCAV_Inv,Book_Value,Tangible_Book_Value,ROE,Price_Book_Ratio,Price_Tangible_Book_Ratio,Price_NCAV_Ratio,Price_NCAV_Inv_Ratio,PE_Ratio,PE_Ratio_TTM
0,1846.HK,2025,2025-12-31,HKD,HKD,869.0m,2.0b,776.7m,400.0m,667.8m,...,108.9m,1.3b,884.5m,4.36%,0.68x,0.98x,7.98x,7.98x,15.96x,15.96x
1,1846.HK,2024,2024-12-31,HKD,HKD,1.3b,1.6b,713.6m,283.7m,473.7m,...,239.9m,1.1b,842.0m,7.53%,1.12x,1.50x,5.26x,5.26x,15.34x,N/A
2,1846.HK,2023,2023-12-31,HKD,HKD,1.7b,1.8b,787.9m,308.7m,588.1m,...,199.8m,1.2b,856.9m,11.57%,1.45x,1.98x,8.49x,8.49x,12.92x,N/A
3,1846.HK,2022,2022-12-31,HKD,HKD,1.7b,1.5b,838.3m,219.7m,497.9m,...,340.4m,1.0b,823.4m,8.82%,1.65x,2.09x,5.06x,5.06x,19.25x,N/A
4,1846.HK,2021,2021-12-31,HKD,HKD,2.5b,N/A,N/A,0,N/A,...,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A,N/A
5,1913.HK,2025,2025-12-31,HKD,EUR,9.9b,11.0b,3.0b,1.9b,6.3b,...,-3.3b,4.7b,2.8b,18.34%,2.13x,3.59x,N/A,N/A,11.65x,11.65x
6,1913.HK,2024,2024-12-31,HKD,EUR,16.5b,8.5b,2.5b,867.9m,4.1b,...,-1.6b,4.4b,3.5b,19.36%,3.79x,4.73x,N/A,N/A,19.66x,N/A
7,2267.T,2025,2025-03-31,JPY,JPY,814.6b,864.3b,377.9b,10.3b,234.8b,...,143.1b,629.5b,619.2b,7.93%,1.29x,1.32x,5.69x,5.69x,17.89x,18.42x
8,3435.T,2025,2025-03-31,JPY,JPY,10.8b,26.6b,15.8b,113.8m,7.8b,...,8.0b,18.8b,18.6b,6.09%,0.57x,0.58x,1.35x,1.35x,9.59x,10.00x
9,6889.HK,2025,2025-03-31,HKD,JPY,44.6b,349.4b,48.0b,7.1b,218.1b,...,-165.3b,131.3b,124.2b,3.05%,0.34x,0.36x,N/A,N/A,11.12x,9.12x
